In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import numba

from numba import njit, prange
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
if not (ROOT / "munik").is_dir():
    ROOT = ROOT.parent
PROJECT_DIR = ROOT / "munik"

CLEAN_DIR = PROJECT_DIR / "analysis" / "output" / "clean"

PRICE_FILE = CLEAN_DIR / "adjusted_close_wide.csv"
QUALITY_FILE = CLEAN_DIR / "stock_quality_report.csv"

print("Python:", sys.version.split()[0])
print("Numba:", numba.__version__)
print("Price file exists:", PRICE_FILE.is_file())
print("Quality file exists:", QUALITY_FILE.is_file())

In [ ]:
@njit
def mean_absolute_distance(a, b):
    total = 0.0

    for i in range(a.size):
        total += abs(a[i] - b[i])

    return total / a.size


a = np.array([0.1, 0.2, -0.3], dtype=np.float64)
b = np.array([0.2, 0.0, -0.3], dtype=np.float64)

result = mean_absolute_distance(a, b)

print("Test distance:", result)
print("Test passed:", np.isclose(result, 0.1))

for _ in tqdm(range(5), desc="Progress bar test"):
    pass

## ۴. خواندن و بررسی اولیه‌ی داده‌های تمیزشده

در این مرحله فایل قیمت‌های تعدیل‌شده و گزارش کیفیت داده را می‌خوانیم.

در جدول قیمت‌ها:
- هر سطر یک روز معاملاتی است.
- هر ستون یک سهم است.
- مقدار هر خانه قیمت تعدیل‌شده‌ی آن سهم در آن روز است.

تعداد روزها، تعداد سهم‌ها و قیمت‌های گمشده را بررسی می‌کنیم.
در این مرحله هیچ مقدار گمشده‌ای را پر نمی‌کنیم.

In [ ]:
# خواندن قیمت‌های تعدیل‌شده
prices_all = pd.read_csv(
    PRICE_FILE,
    parse_dates=["Date"],
    index_col="Date"
).sort_index()

# خواندن گزارش کیفیت داده
quality = pd.read_csv(QUALITY_FILE)

# بررسی تکراری‌نبودن تاریخ‌ها و نام سهم‌ها
assert prices_all.index.is_unique, "Duplicate dates found!"
assert prices_all.columns.is_unique, "Duplicate stock names found!"

print("Trading days:", prices_all.shape[0])
print("Stocks:", prices_all.shape[1])
print("Start date:", prices_all.index.min().date())
print("End date:", prices_all.index.max().date())

# شمارش قیمت‌های گمشده برای هر سهم
missing_per_stock = prices_all.isna().sum()

missing_report = (
    missing_per_stock[missing_per_stock > 0]
    .sort_values(ascending=False)
    .rename("Missing prices")
    .to_frame()
)

print("Total missing prices:", int(missing_per_stock.sum()))

print("\nSample: first 5 days and first 5 stocks")
display(prices_all.iloc[:5, :5])

print("\nStocks with missing prices")
display(missing_report)

## ۵. انتخاب سهم‌های دارای تاریخچه‌ی کامل

برای مقایسه‌ی وضعیت بازار در زمان‌های مختلف، باید در همه‌ی دوره‌ها
همان مجموعه‌ی سهم‌ها حضور داشته باشد.

در این مرحله:
- سهم‌هایی را انتخاب می‌کنیم که در تمام تاریخ‌های موجود قیمت دارند.
- هیچ قیمت گمشده‌ای را با مقدار تخمینی جایگزین نمی‌کنیم.
- مثبت و معتبر بودن قیمت‌های انتخاب‌شده را بررسی می‌کنیم.
- تعداد سهم‌ها در هر بخش اقتصادی را نمایش می‌دهیم.

این مجموعه، جامعه‌ی مورد استفاده‌ی ما برای ادامه‌ی تحلیل است.
جدول اولیه‌ی ۳۰۰ سهم همچنان در متغیر prices_all باقی می‌ماند.

In [ ]:
# شناسایی سهم‌هایی که هیچ قیمت گمشده‌ای ندارند
complete_mask = prices_all.notna().all(axis=0)

# ثبت سهم‌های کنارگذاشته‌شده
excluded_tickers = prices_all.columns[~complete_mask].tolist()

# ساخت جدول قیمت سهم‌های کامل
prices = prices_all.loc[:, complete_mask].copy()

# بررسی معتبر بودن اعداد و مثبت بودن قیمت‌ها
price_values = prices.to_numpy(dtype=np.float64)

assert np.isfinite(price_values).all(), "Invalid prices found!"
assert (price_values > 0).all(), "Zero or negative prices found!"

# اطلاعات بخش اقتصادی هر سهم
assert quality["ticker"].is_unique, "Duplicate tickers in quality report!"

stock_info = (
    quality
    .set_index("ticker")
    .loc[prices.columns, ["sector"]]
    .copy()
)

assert stock_info["sector"].notna().all(), "Missing sector information!"

# تعداد سهم‌های انتخاب‌شده در هر بخش
sector_counts = (
    stock_info["sector"]
    .value_counts()
    .sort_index()
    .rename_axis("Sector")
    .to_frame("Stocks")
)

print("Excluded stocks:", excluded_tickers)
print("Selected stocks:", prices.shape[1])
print("Trading days:", prices.shape[0])
print("Remaining missing prices:", int(prices.isna().sum().sum()))
print("Number of sectors:", len(sector_counts))

display(sector_counts)

## ۶. محاسبه‌ی بازده ساده‌ی روزانه

مقاله به‌جای مقایسه‌ی قیمت‌ها، تغییر نسبی قیمت‌ها را بررسی می‌کند:

r(t) = [S(t+1) - S(t)] / S(t)

مثلاً اگر قیمت از ۱۰۰ به ۱۰۵ برسد، بازده برابر ۰٫۰۵ یا ۵ درصد است.

در این نوت‌بوک، بازده با تاریخ آغاز بازه یعنی t ثبت می‌شود.
بنابراین آخرین روزِ قیمت، بازده جداگانه‌ای ندارد؛ چون قیمت روز بعد را نداریم.

In [ ]:
# تبدیل قیمت‌ها به آرایه‌ی عددی
price_values = prices.to_numpy(dtype=np.float64)

# رابطه‌ی ۳ مقاله
return_values = (
    (price_values[1:] - price_values[:-1])
    / price_values[:-1]
)

# ثبت بازده با تاریخ آغاز بازه
returns = pd.DataFrame(
    return_values,
    index=prices.index[:-1],
    columns=prices.columns
)

assert np.isfinite(return_values).all(), "Invalid returns found!"

print("Return observations:", returns.shape[0])
print("Stocks:", returns.shape[1])
print("First return date:", returns.index.min().date())
print("Last return date:", returns.index.max().date())

display(returns.iloc[:5, :5])

## ۷. نرمال‌سازی محلی بازده‌ها

طبق مقاله، برای هر بازده از ۱۳ مشاهده‌ی اخیر، شامل همان مشاهده،
میانگین و انحراف معیار محلی را محاسبه می‌کنیم:

normalized_return = (return - local_mean) / local_std

در محاسبه‌ی واریانس، مجموع مربع انحراف‌ها بر ۱۳ تقسیم می‌شود.

برای ۱۲ مشاهده‌ی ابتدایی، هنوز سابقه‌ی کافی نداریم.
این دوره‌ی آغازین را از ادامه‌ی تحلیل کنار می‌گذاریم.

اگر انحراف معیار محلی صفر باشد، نتیجه تعریف نشده است.
چنین مواردی را به‌صورت NaN نگه می‌داریم تا در مرحله‌ی بعد گزارش شوند.

از njit برای سریع‌ترکردن حلقه‌های محاسباتی استفاده می‌کنیم.

In [ ]:
@njit
def local_normalize_numba(values, window):
    n_days, n_stocks = values.shape

    result = np.full(
        (n_days, n_stocks),
        np.nan,
        dtype=np.float64
    )

    for j in range(n_stocks):
        for t in range(window - 1, n_days):

            # میانگین ۱۳ مشاهده‌ی اخیر
            local_mean = 0.0

            for q in range(t - window + 1, t + 1):
                local_mean += values[q, j]

            local_mean /= window

            # واریانس ۱۳ مشاهده‌ی اخیر
            local_variance = 0.0

            for q in range(t - window + 1, t + 1):
                difference = values[q, j] - local_mean
                local_variance += difference * difference

            local_variance /= window

            if local_variance > 0.0:
                result[t, j] = (
                    (values[t, j] - local_mean)
                    / np.sqrt(local_variance)
                )

    return result


LOCAL_N = 13

normalized_values = local_normalize_numba(
    np.ascontiguousarray(return_values),
    LOCAL_N
)

# کنارگذاشتن فقط ۱۲ مشاهده‌ی آغازین
normalized_all = pd.DataFrame(
    normalized_values[LOCAL_N - 1:],
    index=returns.index[LOCAL_N - 1:],
    columns=returns.columns
)

# گزارش موارد تعریف‌نشده پس از دوره‌ی آغازین
invalid_counts = (
    normalized_all.isna().sum()
    .rename("Undefined normalized returns")
)

print("Normalized observations:", len(normalized_all))

display(
    invalid_counts[invalid_counts > 0].to_frame()
)

## ۸. ساخت پنجره‌های دوماهه‌ی تقویمی

برای تمام ماتریس‌ها، یک مجموعه‌ی ثابت از سهم‌ها لازم داریم.
سهم‌هایی که بعد از دوره‌ی آغازین، بازده نرمال‌شده‌ی تعریف‌نشده دارند،
از مجموعه‌ی تحلیل کنار گذاشته می‌شوند. جدول قیمت اولیه حفظ می‌شود.

مطابق دوره‌های شکل ۶ مقاله، هر سال را به شش قسمت تقسیم می‌کنیم:

۱. ژانویه و فوریه
۲. مارس و آوریل
۳. مه و ژوئن
۴. ژوئیه و اوت
۵. سپتامبر و اکتبر
۶. نوامبر و دسامبر

پنجره‌ها هم‌پوشانی ندارند و تعداد روزهای معاملاتی آن‌ها لزوماً برابر نیست.

پنجره‌ی اول به‌دلیل نداشتن داده‌ی پیش از سال ۱۹۹۲،
۱۲ مشاهده‌ی آغازین را ندارد. این محدودیت را در جدول ثبت می‌کنیم.

In [ ]:
# سهم‌های معتبر پس از نرمال‌سازی
valid_mask = normalized_all.notna().all(axis=0)

normalization_excluded = (
    normalized_all.columns[~valid_mask].tolist()
)

normalized_returns = normalized_all.loc[:, valid_mask].copy()

model_tickers = normalized_returns.columns.tolist()
model_stock_info = stock_info.loc[model_tickers].copy()

assert np.isfinite(
    normalized_returns.to_numpy()
).all(), "Invalid normalized values remain!"

print("Excluded after normalization:", normalization_excluded)
print("Stocks used for correlations:", len(model_tickers))

# ساخت پنجره‌ها
window_arrays = []
window_records = []

first_year = prices.index.min().year
last_year = prices.index.max().year

for year in range(first_year, last_year + 1):
    for month in [1, 3, 5, 7, 9, 11]:

        start = pd.Timestamp(year=year, month=month, day=1)
        stop = start + pd.DateOffset(months=2)

        block = normalized_returns.loc[
            (normalized_returns.index >= start)
            & (normalized_returns.index < stop)
        ]

        if block.empty:
            continue

        original_block = returns.loc[
            (returns.index >= start)
            & (returns.index < stop)
        ]

        window_arrays.append(
            np.ascontiguousarray(
                block.to_numpy(dtype=np.float64)
            )
        )

        window_records.append({
            "period": f"{year}-T{(month + 1) // 2}",
            "period_start": start,
            "period_end": stop - pd.Timedelta(days=1),
            "n_returns": len(block),
            "warmup_removed": len(original_block) - len(block)
        })

window_info = pd.DataFrame(window_records)

# هیچ مشاهده‌ی معتبر دیگری نباید گم شده باشد
assert window_info["n_returns"].sum() == len(normalized_returns)

print("Number of windows:", len(window_arrays))

display(window_info.head(6))

## ۹. محاسبه‌ی ماتریس‌های همبستگی

برای هر پنجره، همبستگی پیرسون بین بازده‌های نرمال‌شده‌ی سهم‌ها
را محاسبه می‌کنیم.

هر ماتریس، ساختار ارتباط سهم‌ها در یک دوره‌ی دوماهه را نشان می‌دهد.

- مقدار نزدیک ۱: حرکت هم‌جهت
- مقدار نزدیک منفی ۱: حرکت خلاف‌جهت
- مقدار نزدیک صفر: ارتباط خطی ضعیف

قطر اصلی باید برابر ۱ و ماتریس باید متقارن باشد.

از tqdm برای نمایش پیشرفت ساخت ماتریس‌ها استفاده می‌کنیم.

In [ ]:
n_windows = len(window_arrays)
n_stocks = len(model_tickers)

C = np.empty(
    (n_windows, n_stocks, n_stocks),
    dtype=np.float64
)

for i in tqdm(range(n_windows), desc="Correlation matrices"):
    block = window_arrays[i]

    # همبستگی برای سری کاملاً ثابت تعریف نمی‌شود
    assert (block.std(axis=0) > 0).all(), (
        f"Constant series in {window_info.iloc[i]['period']}"
    )

    C[i] = np.corrcoef(block, rowvar=False)

# بررسی صحت ماتریس‌ها
assert np.isfinite(C).all(), "Invalid correlations found!"

assert np.allclose(
    C, C.transpose(0, 2, 1),
    atol=1e-10,
    rtol=0
), "Matrices are not symmetric!"

assert np.allclose(
    np.diagonal(C, axis1=1, axis2=2),
    1.0,
    atol=1e-10,
    rtol=0
), "Invalid diagonal!"

print("Correlation matrices shape:", C.shape)
print("All checks passed.")

# نمایش اولین ماتریس
plt.figure(figsize=(7, 6))

plt.imshow(
    C[0],
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)

plt.colorbar(label="Correlation")
plt.title(f"Correlation matrix: {window_info.iloc[0]['period']}")
plt.xlabel("Stock index")
plt.ylabel("Stock index")

plt.show()

## ۱۰. محاسبه‌ی فاصله‌ی وضعیت‌های بازار

فاصله‌ی دو وضعیت، میانگین قدرمطلق اختلاف درایه‌های دو ماتریس همبستگی است:

distance = mean(abs(C1 - C2))

عدد کوچک‌تر یعنی ساختار همبستگی دو دوره شبیه‌تر است.

برای افزایش سرعت:
- محاسبات عددی با njit اجرا می‌شوند.
- از تقارن ماتریس‌ها استفاده می‌کنیم.
- وزن درایه‌ها طوری محاسبه می‌شود که فاصله با میانگین روی کل ماتریس برابر بماند.

ماتریس D فاصله‌ی هر یک از ۱۱۴ دوره با تمام دوره‌های دیگر را ذخیره می‌کند.

In [ ]:
import sys

ENGINE_FILE = (
    PROJECT_DIR / "analysis" / "market_states_numba.py"
)

assert ENGINE_FILE.is_file(), "Clustering engine file not found!"

sys.path.insert(0, str(ENGINE_FILE.parent))

from market_states_numba import (
    matrix_features,
    pairwise_distances
)

# نمایش فشرده‌ی ماتریس‌ها همراه با وزن‌های صحیح
X, feature_weights = matrix_features(C)

# فاصله‌ها با njit و نمایش پیشرفت
D = pairwise_distances(X, feature_weights)

# بررسی صحت
assert np.isfinite(D).all()
assert np.allclose(D, D.T, atol=1e-12)
assert np.allclose(np.diag(D), 0.0)

# مقایسه با محاسبه‌ی مستقیم روی کل ماتریس
for i, j in [(0, 1), (0, len(C) - 1)]:
    direct_distance = np.mean(np.abs(C[i] - C[j]))

    assert np.isclose(
        D[i, j],
        direct_distance,
        atol=1e-12
    )

print("Feature array shape:", X.shape)
print("Distance matrix shape:", D.shape)
print("Distance checks passed.")

## ۱۱. نمایش فاصله‌ی دوره‌های دوماهه

خانه‌ی (i, j) نشان می‌دهد ساختار همبستگی دوره‌ی i
چقدر با دوره‌ی j تفاوت دارد.

- رنگ روشن: فاصله‌ی کمتر و شباهت بیشتر
- رنگ تیره: فاصله‌ی بیشتر و شباهت کمتر
- قطر اصلی: صفر، چون هر دوره با خودش یکسان است

این نمودار مربوط به ۱۱۴ پنجره‌ی غیرهم‌پوشانِ خوشه‌بندی است.
برای بازسازی پنل چپ شکل ۱ مقاله، بعداً پنجره‌های لغزان را جداگانه محاسبه می‌کنیم.

In [ ]:
OUT_DIR = PROJECT_DIR / "analysis" / "notebook_paper_v2"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ticks = np.arange(0, len(window_info), 12)

years = (
    window_info.iloc[ticks]["period_start"]
    .dt.year.astype(str)
    .tolist()
)

fig, ax = plt.subplots(figsize=(9, 8))

image = ax.imshow(
    D,
    cmap="Blues",
    origin="lower",
    interpolation="nearest",
    vmin=0
)

ax.set_xticks(ticks)
ax.set_xticklabels(years, rotation=45)

ax.set_yticks(ticks)
ax.set_yticklabels(years)

ax.set_xlabel("Period")
ax.set_ylabel("Period")
ax.set_title("Distance between disjoint two-month periods")

fig.colorbar(
    image,
    ax=ax,
    label="Mean absolute correlation difference"
)

fig.tight_layout()
fig.savefig(
    OUT_DIR / "disjoint_period_distances.png",
    dpi=200
)

plt.show()

## ۱۲. ساخت درخت کامل خوشه‌بندی

در ابتدا تمام دوره‌ها در یک گروه قرار می‌گیرند.

برای تقسیم هر گروه:
۱. تمام جفت‌های ممکنِ مراکز اولیه بررسی می‌شوند.
۲. دوره‌ها به مرکز نزدیک‌تر اختصاص پیدا می‌کنند.
۳. مرکز هر زیرگروه با میانگین ماتریس‌های آن محاسبه می‌شود.
۴. اختصاص دوره‌ها تا ثابت‌شدن عضویت‌ها تکرار می‌شود.
۵. بهترین تقسیم با کمترین میانگین مربع فاصله تا مراکز انتخاب می‌شود.

در این پیاده‌سازی، فاصله‌ی رابطه‌ی ۲ مقاله برای خوشه‌بندی استفاده می‌شود.

درخت را تا رسیدن به گروه‌های تک‌عضوی می‌سازیم؛
سپس می‌توانیم آستانه‌های مختلف را بدون تکرار جست‌وجوی مراکز بررسی کنیم.

نتیجه‌ی تقسیم‌های تکراری ذخیره می‌شود تا محاسبه سریع‌تر شود.
همه‌ی جفت‌های اولیه همچنان بررسی می‌شوند.

In [ ]:
import json
from time import perf_counter

from market_states_numba import (
    complete_tree,
    cut_at_threshold,
    threshold_sensitivity
)

def save_tree_progress(nodes):
    """ذخیره‌ی پیشرفت پس از تکمیل هر گره."""
    path = OUT_DIR / "tree_progress.json"

    path.write_text(
        json.dumps(nodes, indent=2),
        encoding="utf-8"
    )


start_time = perf_counter()

tree_nodes = complete_tree(
    X,
    feature_weights,
    D,
    progress=True,
    checkpoint=save_tree_progress
)

elapsed = perf_counter() - start_time

# ذخیره‌ی درخت کامل
(OUT_DIR / "complete_tree.json").write_text(
    json.dumps(tree_nodes, indent=2),
    encoding="utf-8"
)

print("Tree nodes:", len(tree_nodes))
print("Runtime in seconds:", round(elapsed, 2))

## ۱۳. بررسی آستانه و استخراج هشت حالت بازار

ابتدا درخت را با آستانه‌ی اصلی مقاله، یعنی 0.1465، برش می‌دهیم.

سپس بررسی می‌کنیم کدام بازه‌ی آستانه‌ها روی داده‌ی فعلی
دقیقاً هشت گروه تولید می‌کند.

آستانه‌ی هشت‌حالته از همین داده انتخاب شده است؛
بنابراین آن را به‌عنوان آستانه‌ی تنظیم‌شده‌ی پروژه گزارش می‌کنیم.

شماره‌ی حالت‌ها صرفاً برچسب است.
حالت شماره‌ی ۸ ما لزوماً همان حالت شماره‌ی ۸ مقاله نیست؛
برای مقایسه باید ساختار همبستگی و دوره‌های زمانی آن‌ها بررسی شود.

In [ ]:
PAPER_THRESHOLD = 0.1465

# نتیجه با آستانه‌ی منتشرشده در مقاله
labels_paper = cut_at_threshold(
    tree_nodes,
    PAPER_THRESHOLD
)

print(
    "States at paper threshold:",
    len(np.unique(labels_paper))
)

# بررسی تعداد حالت‌ها در آستانه‌های مختلف
sensitivity = threshold_sensitivity(tree_nodes)

eight_options = sensitivity.loc[
    sensitivity["states"] == 8
].copy()

assert not eight_options.empty, (
    "No eight-state threshold interval found. "
    "Send the sensitivity table for inspection."
)

# انتخاب بازه‌ی هشت‌حالته‌ی نزدیک‌تر به آستانه‌ی مقاله
selected_row = eight_options.iloc[
    np.argmin(
        np.abs(
            eight_options["threshold"].to_numpy()
            - PAPER_THRESHOLD
        )
    )
]

CALIBRATED_THRESHOLD = float(selected_row["threshold"])

labels_8 = cut_at_threshold(
    tree_nodes,
    CALIBRATED_THRESHOLD
)

assert len(np.unique(labels_8)) == 8

# جدول نتیجه‌ی هر دوره
state_results = window_info.copy()

state_results["state_paper_threshold"] = labels_paper
state_results["state_calibrated_8"] = labels_8

# ذخیره‌ی خروجی‌ها
sensitivity.to_csv(
    OUT_DIR / "threshold_sensitivity.csv",
    index=False
)

state_results.to_csv(
    OUT_DIR / "state_assignments.csv",
    index=False
)

print("Calibrated threshold:", CALIBRATED_THRESHOLD)
print("States at calibrated threshold:", len(np.unique(labels_8)))

print("\nEight-state threshold interval:")
display(eight_options)

print("\nNumber of periods in each state:")
display(
    pd.Series(labels_8, name="State")
    .value_counts()
    .sort_index()
    .rename_axis("State")
    .to_frame("Periods")
)

## ۱۴. مرتب‌سازی سهم‌ها بر اساس بخش اقتصادی

برای مقایسه‌ی ساختار ماتریس‌ها با مقاله، سهم‌ها را بر اساس
بخش اقتصادی مرتب می‌کنیم:

E → M → I → CD → CS → H → F → IT → C → U

این ترتیب در تمام نمودارها یکسان خواهد بود.

خانه‌های یک بخش روی قطر اصلی، همبستگی داخل آن بخش را نشان می‌دهند.
بلوک‌های خارج از قطر، همبستگی بین دو بخش را نشان می‌دهند.

مرتب‌سازی فقط ترتیب نمایش سهم‌ها را تغییر می‌دهد؛
مقادیر همبستگی و نتیجه‌ی خوشه‌بندی تغییر نمی‌کنند.

In [ ]:
assert "labels_8" in globals(), "Run cells 10–13 first!"

PLOT_ENGINE_FILE = (
    PROJECT_DIR / "analysis" / "paper_state_figures.py"
)

assert PLOT_ENGINE_FILE.is_file(), "Plotting file not found!"

from paper_state_figures import (
    make_sector_layout,
    compute_state_matrices,
    plot_figure2,
    plot_figure3
)

layout = make_sector_layout(
    model_tickers,
    model_stock_info
)

print("Sector order:", layout["sectors"])
print("Stocks:", len(layout["tickers"]))

display(
    layout["counts"].rename_axis("Sector").to_frame("Stocks")
)

## ۱۵. محاسبه‌ی ماتریس متوسط حالت‌ها و میانگین کل

هر حالت شامل چند دوره‌ی دوماهه است.

ماتریس یک حالت را با میانگین‌گیری از ماتریس‌های دوره‌های
عضو همان حالت به دست می‌آوریم:

State matrix = average of matrices assigned to that state

میانگین کل را نیز از تمام ۱۱۴ ماتریس دوره‌ها محاسبه می‌کنیم.

چون تعداد دوره‌های حالت‌ها متفاوت است، میانگین کل را نباید
با میانگین ساده‌ی هشت ماتریس حالت‌ها اشتباه بگیریم.

در این محاسبه، هر دوره‌ی دوماهه وزن یکسانی دارد.

In [ ]:
state_centers, overall_C, state_period_counts = (
    compute_state_matrices(
        C,
        labels_8
    )
)

state_ids = sorted(state_centers)

assert len(state_ids) == 8
assert overall_C.shape == (len(model_tickers), len(model_tickers))

# اختلاف هر حالت با میانگین کل
state_differences = {
    state: state_centers[state] - overall_C
    for state in state_ids
}

print("State matrices:", len(state_centers))
print("Matrix shape:", overall_C.shape)

display(
    pd.Series(state_period_counts, name="Periods")
    .rename_axis("State")
    .to_frame()
)

# ذخیره‌ی ماتریس‌ها و اطلاعات ترتیب سهم‌ها
np.savez_compressed(
    OUT_DIR / "state_matrices.npz",
    state_ids=np.array(state_ids),
    state_centers=np.stack([
        state_centers[state] for state in state_ids
    ]),
    overall=overall_C,
    differences=np.stack([
        state_differences[state] for state in state_ids
    ]),
    tickers=np.array(model_tickers),
    sector_permutation=layout["permutation"]
)

print("State matrices saved.")

## ۱۶. نمایش هشت حالت، درخت خوشه‌بندی و میانگین کل

این شکل شامل ده پنل است:

(a) تا (h): ماتریس متوسط هشت حالت بازار
(i): درخت ارتباط هشت حالت
(j): ماتریس میانگین کل

سهم‌ها در تمام ماتریس‌ها با ترتیب یکسانِ بخش‌های اقتصادی نمایش داده می‌شوند.

درخت از همان درخت خوشه‌بندی محاسبه‌شده استخراج می‌شود.
برای رسم آن، مراکز هشت حالت را دوباره خوشه‌بندی نمی‌کنیم.

طول هر شاخه بر اساس فاصله‌ی مرکز فرزند از مرکز والد محاسبه می‌شود.

مقیاس رنگ تمام ماتریس‌ها یکسان و از منفی ۱ تا مثبت ۱ است:
- آبی: همبستگی مثبت
- سفید: نزدیک صفر
- قرمز: همبستگی منفی

این نمودار، حالت‌های حاصل از آستانه‌ی تنظیم‌شده‌ی پروژه را نشان می‌دهد.

In [ ]:
fig2 = plot_figure2(
    correlations=C,
    labels=labels_8,
    tree_nodes=tree_nodes,
    threshold=CALIBRATED_THRESHOLD,
    layout=layout,
    output_path=OUT_DIR / "figure2_states.png"
)

plt.show()

print("Saved:")
print(OUT_DIR / "figure2_states.png")
print(OUT_DIR / "figure2_states.svg")

## ۱۷. اختلاف ماتریس هر حالت با میانگین کل

برای مشخص‌شدن تفاوت ساختاری حالت‌ها، از ماتریس هر حالت
ماتریس میانگین کل را کم می‌کنیم:

Difference = State matrix - Overall average matrix

در این نمودار:
- آبی: همبستگی بیشتر از میانگین کل
- قرمز: همبستگی کمتر از میانگین کل
- سفید: نزدیک میانگین کل

قرمز بودن در این شکل لزوماً به معنی همبستگی منفی نیست؛
ممکن است همبستگی مثبت باشد، اما از میانگین کل کمتر باشد.

ترتیب سهم‌ها و مقیاس رنگ، همان ترتیب و مقیاس شکل ۲ است.

In [ ]:
fig3 = plot_figure3(
    centers=state_centers,
    overall=overall_C,
    layout=layout,
    output_path=OUT_DIR / "figure3_differences.png"
)

plt.show()

print("Saved:")
print(OUT_DIR / "figure3_differences.png")
print(OUT_DIR / "figure3_differences.svg")

## ۱۸. بررسی عددی نتایج خوشه‌بندی

برای بررسی نتایج، فقط به رنگ نمودارها اکتفا نمی‌کنیم.

برای هر حالت، موارد زیر را محاسبه می‌کنیم:
- تعداد دوره‌های عضو
- میانگین همبستگی جفت سهم‌ها
- میانگین همبستگی داخل بخش‌های اقتصادی
- میانگین همبستگی بین بخش‌های اقتصادی
- سهم همبستگی‌های منفی در ماتریس متوسط

قطر اصلی ماتریس در این آمار استفاده نمی‌شود؛
چون همبستگی هر سهم با خودش همیشه ۱ است.

همچنین با معیار سیلوئت، جدایی گروه‌ها را بررسی می‌کنیم:
- نزدیک ۱: جدایی بیشتر گروه‌ها
- نزدیک صفر: هم‌پوشانی بیشتر
- مقدار منفی: احتمال قرارگیری نامناسب برخی اعضا

این معیار به‌تنهایی برای تعیین بهترین تعداد حالت کافی نیست.

In [ ]:
EXTRA_ENGINE_FILE = (
    PROJECT_DIR / "analysis" / "paper_remaining_figures.py"
)

assert EXTRA_ENGINE_FILE.is_file(), "Extra plotting file not found!"

from paper_remaining_figures import (
    state_diagnostics,
    silhouette_from_distances,
    plot_timeline,
    rolling_correlation_histograms,
    plot_crisis_histograms,
    plot_complete_tree
)

diagnostic_table = state_diagnostics(
    correlations=C,
    labels=labels_8,
    stock_info=model_stock_info,
    tickers=model_tickers,
    window_info=window_info,
    distance_matrix=D
)

cluster_quality = pd.DataFrame({
    "Analysis": [
        "Paper threshold",
        "Calibrated eight states"
    ],
    "States": [
        len(np.unique(labels_paper)),
        len(np.unique(labels_8))
    ],
    "Mean silhouette": [
        silhouette_from_distances(D, labels_paper).mean(),
        silhouette_from_distances(D, labels_8).mean()
    ]
})

display(diagnostic_table.round(4))
display(cluster_quality.round(4))

diagnostic_table.to_csv(
    OUT_DIR / "state_diagnostics.csv",
    index=False
)

cluster_quality.to_csv(
    OUT_DIR / "cluster_quality_comparison.csv",
    index=False
)

## ۱۹. تغییر حالت بازار در طول زمان

هر خط افقی کوتاه، یک دوره‌ی دوماهه را نشان می‌دهد.

- محور افقی: زمان
- محور عمودی: شماره‌ی حالت بازار

اگر چند دوره‌ی متوالی در یک حالت باشند،
خط‌های آن‌ها در امتداد هم قرار می‌گیرند.

بازار می‌تواند از یک حالت خارج شود و بعداً دوباره به همان حالت برگردد.

شماره‌ی حالت فقط یک برچسب است و میزان خطر یا شدت بحران را نشان نمی‌دهد.

In [ ]:
fig4 = plot_timeline(
    window_info=window_info,
    labels=labels_8,
    threshold=CALIBRATED_THRESHOLD,
    output_path=OUT_DIR / "figure4_timeline.png"
)

plt.show()

print("Saved:")
print(OUT_DIR / "figure4_timeline.png")
print(OUT_DIR / "figure4_timeline.svg")

## ۲۰. تغییر توزیع همبستگی‌ها در دوره‌ی بحران

برای بررسی جزئی‌تر بحران، از سپتامبر ۲۰۰۷ تا مارس ۲۰۰۹
برای هر روز معاملاتی یک ماتریس همبستگی جدید محاسبه می‌کنیم.

هر ماتریس بر اساس دو ماه تقویمی اخیرِ بازده‌های نرمال‌شده ساخته می‌شود.
بنابراین این پنجره‌ها لغزان و هم‌پوشان‌اند.

در هر ماتریس:
- همبستگی هر جفت سهم را فقط یک‌بار برمی‌داریم.
- قطر اصلی را کنار می‌گذاریم.
- توزیع ضرایب همبستگی را محاسبه می‌کنیم.

نمودار اول، پایان سپتامبر و پایان دسامبر ۲۰۰۸ را مقایسه می‌کند.
نمودار سه‌بعدی، تغییر توزیع در طول دوره‌ی بحران را نشان می‌دهد.

انتخاب این اجرا برای هیستوگرام:
- ۱۰۰ بازه بین منفی ۱ و مثبت ۱
- ارتفاع برابر سهم جفت‌ها در هر بازه
- نمایش لگاریتمی ارتفاع در نمودار سه‌بعدی

تعداد بازه‌های هیستوگرام در مقاله دقیقاً مشخص نشده است.
وجود دو قله را باید از نتیجه بررسی کنیم؛ آن را از پیش فرض نمی‌کنیم.

In [ ]:
crisis_dates, bin_edges, hist_probabilities, crisis_summary = (
    rolling_correlation_histograms(
        normalized_returns,
        start="2007-09-01",
        end="2009-03-31",
        bins=100,
        progress=True
    )
)

fig5, selected_months = plot_crisis_histograms(
    dates=crisis_dates,
    edges=bin_edges,
    probabilities=hist_probabilities,
    summary=crisis_summary,
    output_path=OUT_DIR / "figure5_crisis_distributions.png"
)

plt.show()

print("Daily windows:", len(crisis_dates))

display(
    selected_months[
        [
            "date",
            "n_returns",
            "mean_correlation",
            "std_correlation",
            "negative_fraction"
        ]
    ].round(4)
)

crisis_summary.to_csv(
    OUT_DIR / "crisis_correlation_summary.csv",
    index=False
)

np.savez_compressed(
    OUT_DIR / "crisis_histograms.npz",
    dates=crisis_dates.to_numpy(),
    edges=bin_edges,
    probabilities=hist_probabilities
)

## ۲۱. نمایش درخت کامل تقسیم دوره‌ها

در شکل ۲، درخت فقط تا هشت حالت نمایش داده شد.
اینجا تمام تقسیم‌ها تا گروه‌های تک‌عضوی نمایش داده می‌شوند.

درخت شامل:
- ۱۱۴ برگ، برای ۱۱۴ دوره‌ی دوماهه
- ۱۱۳ گره‌ی تقسیم
- در مجموع ۲۲۷ گره

برچسبی مثل 92-1 یعنی دوره‌ی اول سال ۱۹۹۲، یعنی ژانویه و فوریه.
برچسب 92-2 یعنی مارس و آوریل همان سال.

عددهای داخل کادر، محل گروه‌های هشت‌حالته‌ی تنظیم‌شده را مشخص می‌کنند.

طول هر شاخه، فاصله‌ی مرکز فرزند از مرکز والد است.
محور افقی زمان نیست؛ فاصله‌ی تجمعی مراکز در مسیر درخت است.

In [ ]:
fig6 = plot_complete_tree(
    correlations=C,
    labels=labels_8,
    tree_nodes=tree_nodes,
    threshold=CALIBRATED_THRESHOLD,
    window_info=window_info,
    output_path=OUT_DIR / "figure6_complete_tree.png"
)

plt.show()

print("Saved:")
print(OUT_DIR / "figure6_complete_tree.png")
print(OUT_DIR / "figure6_complete_tree.svg")

## ۲۲. آماده‌سازی پنجره‌های لغزان روزانه

برای خوشه‌بندی، از ۱۱۴ دوره‌ی دوماهه‌ی غیرهم‌پوشان استفاده کردیم.

برای نمودار فاصله‌ی کل دوره، پنجره را هر روز جلو می‌بریم:
در هر تاریخ، همبستگی دو ماه تقویمی اخیر را محاسبه می‌کنیم.

این پنجره‌ها هم‌پوشانی دارند.

برای داشتن دو ماه کامل از بازده‌های نرمال‌شده،
اولین تاریخ قابل استفاده در داده‌ی ما ۲۰ مارس ۱۹۹۲ است.

پارامتر stride=1 یعنی تمام روزهای معاملاتی حفظ می‌شوند.
در این مرحله نمونه‌برداری هفتگی یا ماهانه انجام نمی‌دهیم.

این تحلیل، نتیجه‌ی خوشه‌بندی قبلی را تغییر نمی‌دهد.

In [ ]:
ROLLING_ENGINE_FILE = (
    PROJECT_DIR / "analysis" / "paper_rolling_analysis.py"
)

assert ROLLING_ENGINE_FILE.is_file(), "Rolling analysis file not found!"

from paper_rolling_analysis import (
    rolling_plan,
    build_rolling_dataset,
    compute_all_rolling_distances,
    plot_rolling_similarity,
    plot_eigenvalue_analysis
)

ROLL_DIR = OUT_DIR / "rolling_daily"

rolling_settings = rolling_plan(
    normalized_returns,
    stride=1
)

print("Daily windows:", rolling_settings["windows"])
print("Stocks:", rolling_settings["stocks"])
print("Matrix pairs:", rolling_settings["pairs"])

print(
    "First date:",
    rolling_settings["dates"][0].date()
)

print(
    "Last date:",
    rolling_settings["dates"][-1].date()
)

disk_gb = (
    rolling_settings["feature_bytes"]
    + rolling_settings["distance_bytes"]
) / 1e9

print("Main array files, approximately:", round(disk_gb, 2), "GB")

## ۲۳. محاسبه‌ی ساختار همبستگی برای تمام روزها

برای هر پنجره‌ی دوماهه‌ی لغزان:
۱. ماتریس همبستگی را محاسبه می‌کنیم.
۲. درایه‌های لازم را با استفاده از تقارن ذخیره می‌کنیم.
۳. بزرگ‌ترین مقدار ویژه‌ی ماتریس را محاسبه می‌کنیم.

محاسبات با دقت float64 انجام می‌شوند.

برای کنترل مصرف حافظه، آرایه‌های بزرگ روی دیسک ذخیره می‌شوند
و به‌صورت memory-mapped در دسترس قرار می‌گیرند.

پیشرفت نیز ذخیره می‌شود تا اجرای قطع‌شده قابل ادامه باشد.

اگر داده‌ی ورودی تغییر کند، کد اجازه نمی‌دهد
نتایج قدیمی با داده‌ی جدید مخلوط شوند.

In [ ]:
rolling_data = build_rolling_dataset(
    normalized_returns=normalized_returns,
    output_dir=ROLL_DIR,
    stride=1,
    progress=True
)

print(
    "Stored feature array shape:",
    rolling_data["features"].shape
)

print(
    "Largest-eigenvalue array shape:",
    rolling_data["eigenvalues"].shape
)

rolling_window_table = pd.read_csv(
    ROLL_DIR / "rolling_windows.csv",
    parse_dates=[
        "date",
        "first_return_date",
        "last_return_date"
    ]
)

display(rolling_window_table.head())

## ۲۴. محاسبه‌ی فاصله‌ی تمام وضعیت‌های روزانه

برای هر جفت تاریخ، میانگین قدرمطلق اختلاف
ماتریس‌های همبستگی را محاسبه می‌کنیم.

از njit و پردازش موازی برای افزایش سرعت استفاده می‌شود.

فقط نیمی از جفت‌ها نیاز به محاسبه دارند؛
چون فاصله‌ی تاریخ اول تا دوم با فاصله‌ی دوم تا اول برابر است.

نتایج در چند مرحله ذخیره می‌شوند.
اگر اجرا قطع شود، اجرای دوباره از بخش‌های باقی‌مانده ادامه می‌یابد.

نام خروجی این مرحله D_daily است.
ماتریس D مربوط به ۱۱۴ دوره‌ی خوشه‌بندی حفظ می‌شود.

In [ ]:
D_daily = compute_all_rolling_distances(
    rolling=rolling_data,
    max_threads=8,
    chunk_rows=8,
    progress=True
)

print("Daily distance matrix shape:", D_daily.shape)

print(
    "Maximum diagonal error:",
    np.max(np.abs(np.diag(D_daily)))
)

## ۲۵. نمایش فاصله‌ی روزانه و روش جایگزین مقدار ویژه

نمودار اول شامل دو پنل است:

- پنل چپ: فاصله‌ی ساختار همبستگی در کل دوره‌ی موجود
- پنل راست: بزرگ‌نمایی روزانه‌ی بازه‌ی ۲۰۰۷ تا ۲۰۱۰

پنل راستِ مقاله با داده‌های درون‌روزی TAQ ساخته شده است.
پنل راست این پروژه بزرگ‌نمایی داده‌ی روزانه است
و با همین عنوان گزارش می‌شود.

در هر پنل:
- رنگ روشن یعنی فاصله‌ی کمتر
- رنگ تیره یعنی فاصله‌ی بیشتر
- قطر اصلی صفر است

در نمودار بعدی، روش جایگزین مقاله را بررسی می‌کنیم:

alternative_distance(t1, t2)
= abs(largest_eigenvalue(t1) - largest_eigenvalue(t2))

این فاصله فقط یک ویژگی از ماتریس را مقایسه می‌کند.
مقیاس عددی آن با فاصله‌ی میانگین اختلاف ماتریس‌ها متفاوت است.

In [ ]:
# فاصله‌ی ماتریس‌های همبستگی در کل دوره
fig1 = plot_rolling_similarity(
    rolling=rolling_data,
    distance=D_daily,
    output_path=OUT_DIR / "figure1_daily_rolling_similarity.png"
)

plt.show()

# روش جایگزین بر اساس بزرگ‌ترین مقدار ویژه
fig_eigen = plot_eigenvalue_analysis(
    rolling=rolling_data,
    output_path=OUT_DIR / "largest_eigenvalue_alternative.png"
)

plt.show()

print("Saved:")
print(OUT_DIR / "figure1_daily_rolling_similarity.png")
print(OUT_DIR / "figure1_daily_rolling_similarity.svg")
print(OUT_DIR / "largest_eigenvalue_alternative.png")
print(OUT_DIR / "largest_eigenvalue_alternative.svg")